# MediCare Patient Follow-Up Agent

## Setup & Imports

In [3]:
# Install dependencies
!pip install langchain langchain-openai pandas tabulate -q

ERROR: Ignored the following versions that require a different python version: 0.0.1 Requires-Python >=3.8.1,<4.0; 0.0.1rc0 Requires-Python >=3.8.1,<4.0; 0.0.1rc1 Requires-Python >=3.8.1,<4.0; 0.0.2 Requires-Python >=3.8.1,<4.0; 0.0.2.post1 Requires-Python >=3.8.1,<4.0; 0.0.3 Requires-Python >=3.8.1,<4.0; 0.0.4 Requires-Python >=3.8.1,<4.0; 0.0.5 Requires-Python >=3.8.1,<4.0; 0.0.6 Requires-Python >=3.8.1,<4.0; 0.0.7 Requires-Python >=3.8.1,<4.0; 0.0.8 Requires-Python >=3.8.1,<4.0; 0.0.8rc1 Requires-Python >=3.8.1,<4.0; 0.1.0 Requires-Python <4.0,>=3.8.1; 0.1.1 Requires-Python <4.0,>=3.8.1; 0.1.10 Requires-Python <4.0,>=3.8.1; 0.1.11 Requires-Python <4.0,>=3.8.1; 0.1.12 Requires-Python <4.0,>=3.8.1; 0.1.13 Requires-Python <4.0,>=3.8.1; 0.1.14 Requires-Python <4.0,>=3.8.1; 0.1.15 Requires-Python <4.0,>=3.8.1; 0.1.16 Requires-Python <4.0,>=3.8.1; 0.1.17 Requires-Python <4.0,>=3.8.1; 0.1.19 Requires-Python <4.0,>=3.8.1; 0.1.2 Requires-Python <4.0,>=3.8.1; 0.1.20 Requires-Python <4.0,>=3.8

In [2]:
import os
import json
import pandas as pd
from datetime import datetime
from typing import Any

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ── API Key (mask before submission) ──────────────────────────────────────────
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-XXXX-YOUR-KEY-HERE"

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
print("✅ LLM initialised:", llm.model)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Task 1 — Data Loading & Exploratory Analysis

In [ ]:
df = pd.read_csv("patient_data.csv", parse_dates=["last_visit_date", "next_scheduled_visit"])
print(f"Dataset: {df.shape[0]} patients × {df.shape[1]} columns")
df.head(3)

In [ ]:
# Quick EDA
print("=== Diagnosis Distribution ===")
print(df["diagnosis"].value_counts().head(10).to_string())

print("\n=== Missed Appointments ===")
print(df["missed_last_appointment"].value_counts().to_string())

print("\n=== Key Vitals Summary ===")
print(df[["vitals_bp_systolic", "vitals_bp_diastolic", "vitals_heart_rate", "vitals_spo2"]].describe().round(1).to_string())

print("\n=== Days Since Last Visit ===")
print(df["days_since_last_visit"].describe().round(1).to_string())

## Task 2 — Define Agent Tools

In [ ]:
# Clinical thresholds reference
THRESHOLDS = {
    "HbA1c":      {"high": 7.0,   "unit": "%",    "condition": "Type 2 Diabetes"},
    "BNP":        {"high": 100,   "unit": "pg/mL","condition": "Heart Failure"},
    "Creatinine": {"high": 1.2,   "unit": "mg/dL","condition": "CKD"},
    "Hemoglobin": {"low":  12.0,  "unit": "g/dL", "condition": "Anemia"},
    "FEV1":       {"low":  70.0,  "unit": "%",    "condition": "COPD"},
}

@tool
def get_patient_record(patient_id: str) -> str:
    """Retrieve full clinical record for a single patient by patient_id (e.g. 'P0001')."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return f"No patient found with ID {patient_id}."
    return row.to_dict(orient="records")[0].__str__()


@tool
def flag_lab_risk(patient_id: str) -> str:
    """Check whether a patient's latest lab value exceeds clinical thresholds. Returns risk level and explanation."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    lab, val = r["lab_test"], float(r["lab_value"])
    thresh = THRESHOLDS.get(lab)
    if not thresh:
        return f"No threshold defined for lab test '{lab}'."
    if "high" in thresh and val > thresh["high"]:
        return (f"HIGH RISK — {lab} = {val} {thresh['unit']} "
                f"(threshold > {thresh['high']}). Condition: {thresh['condition']}.")
    if "low" in thresh and val < thresh["low"]:
        return (f"HIGH RISK — {lab} = {val} {thresh['unit']} "
                f"(threshold < {thresh['low']}). Condition: {thresh['condition']}.")
    return f"NORMAL — {lab} = {val} {thresh['unit']} within acceptable range."


@tool
def flag_vitals_risk(patient_id: str) -> str:
    """Assess whether a patient's blood pressure, heart rate, or SpO2 are outside safe ranges."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    flags = []
    if r["vitals_bp_systolic"] >= 140 or r["vitals_bp_diastolic"] >= 90:
        flags.append(f"Hypertension: BP {r['vitals_bp_systolic']}/{r['vitals_bp_diastolic']} mmHg")
    if r["vitals_heart_rate"] > 100:
        flags.append(f"Tachycardia: HR {r['vitals_heart_rate']} bpm")
    if r["vitals_spo2"] < 92:
        flags.append(f"Hypoxia: SpO2 {r['vitals_spo2']}%")
    return "VITALS RISKS: " + "; ".join(flags) if flags else "Vitals within normal limits."


@tool
def get_missed_appointment_patients(top_n: int = 10) -> str:
    """Return patients who missed their last appointment, sorted by days since last visit (most overdue first)."""
    missed = (df[df["missed_last_appointment"].str.lower() == "yes"]
              .sort_values("days_since_last_visit", ascending=False)
              .head(top_n)[["patient_id", "patient_name", "diagnosis",
                            "days_since_last_visit", "next_scheduled_visit"]])
    if missed.empty:
        return "No missed appointments found."
    return missed.to_string(index=False)


@tool
def generate_care_action_plan(patient_id: str, risk_summary: str) -> str:
    """Given a patient_id and a plain-text risk_summary, produce a structured follow-up care action plan."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    plan = {
        "patient_id": patient_id,
        "patient_name": r["patient_name"],
        "diagnosis": r["diagnosis"],
        "risk_summary": risk_summary,
        "recommended_actions": "[LLM will fill this in]",
        "priority": "[LLM will fill this in]",
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    return json.dumps(plan, indent=2)


tools = [get_patient_record, flag_lab_risk, flag_vitals_risk,
         get_missed_appointment_patients, generate_care_action_plan]
print(f"✅ {len(tools)} tools registered:", [t.name for t in tools])

## Task 3 — Implement the Agentic Loop

In [ ]:
SYSTEM_PROMPT = """You are MediCare Follow-Up Agent, a clinical AI assistant.
You autonomously review patient records, identify clinical risks, and generate
prioritised care action plans for the care coordination team.

When analysing a patient:
1. Retrieve their record.
2. Check lab value risks.
3. Check vitals risks.
4. Synthesise findings and call generate_care_action_plan with a clear risk summary.
5. In your final answer, output a structured plan with: Priority (HIGH/MEDIUM/LOW),
   key risks, and 3–5 specific recommended actions.

Be concise, clinical, and actionable. Never hallucinate lab values or diagnoses."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,           # shows tool calls in the agentic loop
    max_iterations=8,
    handle_parsing_errors=True,
)
print("✅ AgentExecutor ready")

## Task 4 — Single Patient Analysis

In [ ]:
PATIENT_ID = "P0001"   # Change to any patient_id from the CSV

result = agent_executor.invoke({
    "input": f"Perform a full clinical risk assessment for patient {PATIENT_ID} "
             f"and generate a prioritised follow-up care action plan."
})
print("\n" + "="*60)
print("AGENT OUTPUT")
print("="*60)
print(result["output"])

## Task 5 — Missed Appointment Follow-Up

In [ ]:
# Step 1: Identify top missed-appointment patients
missed_result = agent_executor.invoke({
    "input": "List the top 5 patients who missed their last appointment, "
             "sorted by how overdue they are."
})
print(missed_result["output"])

In [ ]:
# Step 2: Batch analysis — run the agent for each missed-appointment patient
missed_patients = (
    df[df["missed_last_appointment"].str.lower() == "yes"]
    .sort_values("days_since_last_visit", ascending=False)
    .head(5)["patient_id"]
    .tolist()
)
print(f"Analysing {len(missed_patients)} overdue patients: {missed_patients}\n")

batch_results = []
for pid in missed_patients:
    print(f"--- Analysing {pid} ---")
    r = agent_executor.invoke({
        "input": f"Patient {pid} missed their last appointment and is overdue. "
                 f"Assess all clinical risks and generate a follow-up care action plan."
    })
    batch_results.append({"patient_id": pid, "plan": r["output"]})
    print(r["output"])
    print()

In [ ]:
# Step 3: Summary dashboard of all batch results
print("="*60)
print("MISSED APPOINTMENT FOLLOW-UP — SUMMARY DASHBOARD")
print("="*60)
for item in batch_results:
    print(f"\n[{item['patient_id']}]")
    # Extract first 3 lines of each plan for the summary view
    print("\n".join(item["plan"].split("\n")[:6]))
    print("...")